# Session 13 — End-to-End MLOps Pipeline to Deploy and Monitor ML and LLM-Based Applications on GCP

**Goal:** extend the classical-ML pipeline pattern from Session 12 to an
**LLM-based application** — deploying a foundation model endpoint (Vertex AI's
Gemini or a custom fine-tuned model), wrapping it behind a versioned API, and
monitoring quality with metrics that don't apply to a classifier (latency, token
cost, response quality).

## What's different about "MLOps" for LLMs

A classifier's quality is a single number (AUC, accuracy). An LLM application's
quality involves cost per request (billed by token), latency (users notice seconds,
not milliseconds), and *content* quality, which usually needs either human review or
another LLM acting as a judge. The deployment mechanics (endpoint, monitoring,
CI/CD) carry over from the rest of this course; the metrics you monitor do not.

## Prerequisites

Needs a **GCP project** with the Vertex AI Generative AI API enabled — not available
in this sandbox. Complete, correct reference code below.

```bash
pip install google-cloud-aiplatform
```

In [ ]:
PROJECT_ID = "your-gcp-project-id"
REGION = "us-central1"

## Step 1 — Call a hosted foundation model

Vertex AI hosts foundation models (Gemini family) behind a managed endpoint — no
training or deployment step needed for the base model itself, unlike every prior
session in this course.

In [ ]:
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project=PROJECT_ID, location=REGION)
model = GenerativeModel("gemini-1.5-flash")

response = model.generate_content(
    "Summarize the key steps of an MLOps pipeline in 3 bullet points."
)
print(response.text)
print(f"Input tokens: {response.usage_metadata.prompt_token_count}")
print(f"Output tokens: {response.usage_metadata.candidates_token_count}")

## Step 2 — Wrap it as a versioned API (same shape as Session 7)

Same FastAPI pattern used for the classical model in Session 7 — the *interface* an
LLM app exposes doesn't need to look different from a classifier's API, even though
what happens inside does.

In [ ]:
llm_api_code = '''\
from fastapi import FastAPI
from pydantic import BaseModel
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project="your-gcp-project-id", location="us-central1")
model = GenerativeModel("gemini-1.5-flash")

app = FastAPI(title="LLM Summarizer API", version="1.0.0")

class SummarizeRequest(BaseModel):
    text: str
    max_words: int = 50

class SummarizeResponse(BaseModel):
    summary: str
    input_tokens: int
    output_tokens: int

@app.post("/v1/summarize", response_model=SummarizeResponse)
def summarize(request: SummarizeRequest):
    prompt = f"Summarize the following in under {request.max_words} words:\\n\\n{request.text}"
    response = model.generate_content(prompt)
    return SummarizeResponse(
        summary=response.text,
        input_tokens=response.usage_metadata.prompt_token_count,
        output_tokens=response.usage_metadata.candidates_token_count,
    )
'''
with open("session13_llm_api.py", "w") as f:
    f.write(llm_api_code)
print(llm_api_code)

## Step 3 — Deploy a custom fine-tuned model instead (optional path)

If a foundation model isn't enough (Session 8's course-wide fine-tuning example,
`10. NLP/Deep learning/7. LLM Pytorch/Projects/finetuning_llama2.ipynb`, produces a
custom checkpoint), Vertex AI can serve a custom model the same way it serves any
other model artifact — via a `Model` resource and `.deploy()`, same as Session 4.

In [ ]:
from google.cloud import aiplatform

custom_model = aiplatform.Model.upload(
    display_name="finetuned-llama2-summarizer",
    artifact_uri="gs://your-bucket/finetuned-llama2/",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/pytorch-gpu.2-1:latest",
)
endpoint = custom_model.deploy(
    machine_type="n1-standard-8",
    accelerator_type="NVIDIA_TESLA_T4",
    accelerator_count=1,
)
print(f"Custom LLM endpoint: {endpoint.resource_name}")

## Step 4 — Monitor LLM-specific quality signals

Cost and latency are straightforward to log per request; **quality** for open-ended
text usually needs either human labeling of a sample, or an "LLM-as-judge" pattern —
asking a second model to score the first model's output against a rubric.

In [ ]:
def log_llm_call(request_id, prompt, response, latency_ms):
    return {
        "request_id": request_id,
        "input_tokens": response.usage_metadata.prompt_token_count,
        "output_tokens": response.usage_metadata.candidates_token_count,
        "estimated_cost_usd": (
            response.usage_metadata.prompt_token_count * 0.000_000_075
            + response.usage_metadata.candidates_token_count * 0.000_000_30
        ),
        "latency_ms": latency_ms,
    }


def llm_as_judge_score(judge_model, original_text, summary):
    judge_prompt = f'''\
Rate this summary from 1-5 for accuracy and conciseness. Respond with only the number.

Original: {original_text}
Summary: {summary}
'''
    judge_response = judge_model.generate_content(judge_prompt)
    return judge_response.text.strip()

print("log_llm_call() feeds the same BigQuery monitoring table pattern from Session 12.")
print("llm_as_judge_score() gives an automated quality proxy to track over time,")
print("alongside periodic human spot-checks -- never as the only quality signal.")

## Step 5 — CI/CD for prompts

Treat prompt templates like code: version them in Git, and run a small regression
suite (fixed inputs with expected properties, not exact-match outputs) in the CI
pipeline from Session 10 before a prompt change ships.

In [ ]:
prompt_regression_test = '''\
def test_summary_stays_under_word_limit():
    response = client.post("/v1/summarize", json={"text": LONG_ARTICLE, "max_words": 50})
    summary = response.json()["summary"]
    assert len(summary.split()) <= 60  # allow slight overshoot, not unbounded

def test_summary_is_nonempty():
    response = client.post("/v1/summarize", json={"text": LONG_ARTICLE})
    assert len(response.json()["summary"]) > 0
'''
print(prompt_regression_test)

## What to try next

* Add token-budget alerting: if `estimated_cost_usd` summed over a day exceeds a
  threshold, alert the team the same way Session 5's drift detector would.
* Compare hosted Gemini calls against the `finetuning_llama2.ipynb` project's
  self-hosted checkpoint on cost, latency, and quality for your specific use case.
* Session 14 automates retraining/redeployment triggers for the classical-ML case;
  the LLM analogue is usually "re-evaluate the prompt/model against the regression
  suite whenever the underlying foundation model version changes."